# FIT4112 – Lab 3 | Notebook 02: K-Means Clustering

**Bài toán:** Từ RFM, tìm các nhóm khách hàng có hành vi mua tương tự nhau.

**Hai lộ trình chung một notebook:**
- **FOUNDATION (gợi ý HTTT):** đã có hướng dẫn, thử k = 2, 3, 4 với `log1p + StandardScaler`.
- **APPLIED (gợi ý PTPM):** so sánh `log1p + StandardScaler` và `StandardScaler` trực tiếp, thử k = 2, 3, 4, 5.

**Cả hai đều có thể đạt 100 điểm.** Không cần viết K-Means bằng tay.

**Nhắc lại:** K-Means gán mỗi điểm về một tâm cụm, cập nhật tâm và lặp. Khoảng cách giữa các điểm phụ thuộc mạnh vào thang đo đặc trưng.

## Bước 1. Đọc RFM và chọn lộ trình

Nếu chưa chạy Notebook 01, notebook tự dùng `data/starter/customer_rfm.csv` để vẫn học được K-Means.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / "data" / "raw" / "retail_transactions_demo.csv").exists():
            return directory
    raise FileNotFoundError("Không tìm thấy repo FIT4112 Lab 3. Mở notebook từ repo hoặc notebooks/.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục dự án:", ROOT)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

TRACK = "FOUNDATION"  # TODO: có thể đổi thành "APPLIED"
TRACK = TRACK.upper().strip()
assert TRACK in {"FOUNDATION", "APPLIED"}, "TRACK phải là FOUNDATION hoặc APPLIED"

rfm_file = OUT_DIR / "customer_rfm.csv"
if not rfm_file.exists():
    rfm_file = ROOT / "data" / "starter" / "customer_rfm.csv"
    print("Chưa có output Notebook 01 -> dùng dữ liệu RFM starter.")
rfm = pd.read_csv(rfm_file, dtype={"CustomerID": "string"})
features = ["Recency", "Frequency", "Monetary"]
assert rfm["CustomerID"].is_unique
assert rfm[features].gt(0).all().all()
print(f"Lộ trình: {TRACK}; khách hàng: {len(rfm)}")
display(rfm.head())

## Bước 2. Chuẩn hóa và thử số cụm k

- `log1p`: nén sự chênh lệch lớn giữa các giá trị dương.
- `StandardScaler`: đưa đặc trưng về thang tương đương (trung bình 0, độ lệch chuẩn 1).
- `Silhouette`: gần 1 nghĩa là các điểm cùng cụm thường gần nhau hơn so với cụm khác; **không phải** thước đo giá trị kinh doanh.

K-Means luôn dùng cùng `random_state=42` và `n_init=10` để dễ tái lập.

In [ ]:
def transform_rfm(frame, mode):
    x = frame[features].astype(float)
    if mode == "log1p+scaler":
        x = np.log1p(x)
    return StandardScaler().fit_transform(x)

modes = ["log1p+scaler"] if TRACK == "FOUNDATION" else ["log1p+scaler", "scaler-only"]
k_values = [2, 3, 4] if TRACK == "FOUNDATION" else [2, 3, 4, 5]
records = []
for mode in modes:
    x = transform_rfm(rfm, mode)
    for k in k_values:
        labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(x)
        sizes = pd.Series(labels).value_counts()
        score = silhouette_score(x, labels) if 1 < len(sizes) < len(rfm) else np.nan
        records.append({"mode": mode, "k": k, "silhouette": score,
                        "min_cluster_size": int(sizes.min()),
                        "max_cluster_size": int(sizes.max())})
scores = pd.DataFrame(records)
display(scores.round(3))
scores.to_csv(OUT_DIR / "clustering_scores.csv", index=False)

## Bước 3. Chọn k có căn cứ

Ví dụ hướng dẫn: ưu tiên cấu hình có `silhouette` cao nhưng không tạo cụm quá nhỏ (**ít nhất 5 khách hàng** trong dữ liệu demo). Nếu tất cả tạo cụm quá nhỏ, chọn cấu hình có silhouette cao nhất và giải thích.

**TODO:** Quan sát kết quả và sửa `CHOSEN_MODE` / `CHOSEN_K` nếu bạn có lập luận khác. Foundation cũng được phép chủ động điều chỉnh.

In [ ]:
eligible = scores.loc[scores["min_cluster_size"] >= 5]
if eligible.empty:
    eligible = scores
best = eligible.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]
CHOSEN_MODE = str(best["mode"])  # TODO: chọn cấu hình có căn cứ
CHOSEN_K = int(best["k"])        # TODO: chọn số cụm có căn cứ
assert CHOSEN_MODE in modes and CHOSEN_K in k_values
chosen_row = scores[(scores["mode"] == CHOSEN_MODE) & (scores["k"] == CHOSEN_K)]
assert len(chosen_row) == 1
x = transform_rfm(rfm, CHOSEN_MODE)
model = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=10)
labels = model.fit_predict(x)
segments = rfm.copy()
segments["Cluster"] = labels
print("Cấu hình đã chọn:", CHOSEN_MODE, "k =", CHOSEN_K)
print("Silhouette:", round(silhouette_score(x, labels), 3))
display(segments["Cluster"].value_counts().sort_index().rename("Số khách hàng").to_frame())

## Bước 4. Trực quan kết quả trên RFM gốc

Biểu đồ Frequency–Monetary giúp quan sát; thuật toán thực tế chạy trong **không gian ba đặc trưng** nên không kết luận chất lượng cụm chỉ dựa trên hình chiếu 2D.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for cluster_id, group in segments.groupby("Cluster"):
    ax.scatter(group["Frequency"], group["Monetary"], label=f"Cụm {cluster_id}", alpha=0.8)
ax.set_xlabel("Frequency (số hóa đơn)")
ax.set_ylabel("Monetary (tổng chi tiêu)")
ax.set_title("Phân cụm khách hàng (biểu đồ RFM gốc)")
ax.legend()
plt.tight_layout()
plt.show()
segments.to_csv(OUT_DIR / "customer_segments.csv", index=False)
# Ghi lại cấu hình đã chọn để Notebook 04 dùng đúng baseline (không đoán lại chế độ transform).
import json
config = {
    "track": TRACK,
    "feature_mode": CHOSEN_MODE,
    "k": CHOSEN_K,
    "random_state": 42,
    "n_init": 10,
}
(OUT_DIR / "selected_model.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã lưu customer_segments.csv, clustering_scores.csv và selected_model.json")

## TODO / Câu hỏi

**Foundation:** k = 2/3/4 có silhouette và số khách hàng trong cụm thay đổi thế nào? Vì sao cần chuẩn hóa?

**Applied:** so sánh hai cách biến đổi RFM; cấu hình có silhouette cao nhất có luôn dễ diễn giải nhất không?

**Cả hai:** tại sao Cluster 0 không tự động có nghĩa là khách hàng tốt nhất?

**Tiếp theo:** Notebook 03 – Customer Segmentation Insights.